# Colab evaluation launcher (multi-dataset)

Zero-shot evaluation of Kvasir-trained checkpoints on Kvasir-SEG **test**
plus the external datasets (CVC-ClinicDB, CVC-ColonDB, ETIS, CVC-300 when
available), then one aggregate table and per-run tables.

Run the cells top to bottom. Edit **only cell 2** to change what runs.

- `ONLY_RUNS` is required: an empty list stops the notebook with a clear
  message instead of scoring every folder under `RUNS_DIR`.
- `src.evaluate` appends one row per `(run, split, dataset, tta)` to
  `results.csv`; `--force` replaces an existing row in place.
- Before the first write, `results.csv` is copied to a timestamped backup.
- The test split is evaluated once per run and is never used for model
  selection; `src.evaluate` aborts if a test id leaks into train/val.
- Checkpoints and results live on Drive and survive a disconnect.


In [ ]:
from pathlib import Path

# ==== EDIT ONLY THIS CELL ============================================

# Commit/tag/branch to check out. Default: the base commit that contains
# the data.augment fix (33f4b0e is on this branch).
REF = "4ea2fcb"
REPO_URL = "https://github.com/awdtyo/Polyp-Segmentation.git"

# Runs and results live on Drive.
RUNS_DIR = "/content/drive/MyDrive/polyp-runs"
RESULTS_CSV = f"{RUNS_DIR}/results.csv"

# Drive data tree: raw/<dataset>/ and optional processed/<dataset>/.
# I will confirm the exact path before the first real run.
DRIVE_DATA_ROOT = "/content/drive/MyDrive/polyp-data"
DRIVE_ZIP = "/content/drive/MyDrive/kvasir-seg.zip"

# Explicit list of run folder names to evaluate. Empty => stop with a
# clear message instead of scoring every folder under RUNS_DIR.
ONLY_RUNS: list[str] = []

# Datasets to score. "etis" and "cvc_300" only when you add them.
DATASETS: list[str] = ["kvasir_seg", "cvc_clinicdb", "cvc_colondb"]

EVAL_SPLIT = "test"          # Kvasir split; external sets always use all images
TTA_FLIP = False             # pass --tta-flip only when True
FORCE = True                 # replace an existing (run, split, dataset, tta) row
SANITY_CHECK = True          # evaluate each run on Kvasir val first

REPO_DIR = Path("/content/polyp")
LOCAL_DATA_ROOT = "/content/polyp-data"
# =====================================================================


## Setup: mount Drive, get the repo at REF, install missing packages

`GH_TOKEN` is read from Colab Secrets and never printed. The repo is cloned
at `REF` and then reset to a clean copy (`git reset --hard`, `git clean -fdq`)
before checkout, so a rerun cannot silently pick up newer code. The token is
removed from `.git/config` afterwards.


In [ ]:
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
if not token:
    raise RuntimeError(
        "Missing GH_TOKEN. Add it in Colab Secrets (key icon) and enable it "
        "for this notebook."
    )


    def _auth_url(url, secret):
        """Insert a token into an https clone URL without storing it anywhere."""
        if not url.startswith("https://"):
            raise ValueError("REPO_URL must be an https URL")
        return url.replace("https://", f"https://x-access-token:{secret}@", 1)


    def _git(args, check=True):
        result = subprocess.run(["git", *args], capture_output=True, text=True)
        if check and result.returncode != 0:
            message = (result.stderr or result.stdout).replace(token, "***")
            raise RuntimeError(f"git failed: {message}")
        return result

    if (REPO_DIR / ".git").is_dir():
        _git(["-C", str(REPO_DIR), "fetch", "origin", "--tags", "--prune"])
    else:
        _git(["clone", _auth_url(REPO_URL, token), str(REPO_DIR)])
        _git(["-C", str(REPO_DIR), "fetch", "origin", "--tags", "--prune"])

    # Reset to a clean copy at REF before checkout.
    _git(["-C", str(REPO_DIR), "reset", "--hard"])
    _git(["-C", str(REPO_DIR), "clean", "-fdq"])

    has_remote_branch = (
        _git(["-C", str(REPO_DIR), "rev-parse", "--verify", "--quiet", f"origin/{REF}"], check=False).returncode
        == 0
    )
    if has_remote_branch:
        checkout = _git(["-C", str(REPO_DIR), "checkout", "-B", REF, f"origin/{REF}"], check=False)
    else:
        checkout = _git(["-C", str(REPO_DIR), "checkout", REF], check=False)
    if checkout.returncode != 0:
        message = (checkout.stderr or checkout.stdout).replace(token, "***")
        raise RuntimeError(f"Could not check out REF={REF!r}: {message}")

    # Do not leave the token in .git/config.
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
        check=True,
    )
    del token

    # Verify the tree is clean.
    status = _git(["-C", str(REPO_DIR), "status", "--porcelain"], check=False)
    if status.stdout.strip():
        raise RuntimeError(
            f"Repo at {REPO_DIR} is not clean after checkout:\n{status.stdout}"
        )

    head = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print(f"repo ready at {head} (REF={REF})")


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "timm": "timm",
    "albumentations": "albumentations",
    "yaml": "pyyaml",
    "pytest": "pytest",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all required Python packages are already present", flush=True)


## Data: stage processed datasets for DATASETS on the local disk

Kvasir-SEG is built from `DRIVE_ZIP` with `scripts/build_cache.py` at each
run's `data.image_size`, so it is array-identical to the training cache.
Every other dataset is copied from `DRIVE_DATA_ROOT/processed/<name>/` when
present and valid (matching `expected_count` and `meta.json` image size);
otherwise it is prepared from `DRIVE_DATA_ROOT/raw/<name>/` with
`scripts/prepare_datasets.py`.

Staged data lands under `LOCAL_DATA_ROOT` (`/content/polyp-data`), one size-
specific data root per distinct `image_size` so a 256 px and a 352 px run
never share a cache.


In [ ]:
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import yaml

sys.path.insert(0, str(REPO_DIR))
from src.data_registry import get_spec, registry_names  # noqa: E402

# --- Discover runs and their image sizes -------------------------------
runs_root = Path(RUNS_DIR)
if not runs_root.is_dir():
    raise FileNotFoundError(f"No runs folder at {runs_root}.")

candidates = []
for candidate in sorted(runs_root.iterdir()):
    if not candidate.is_dir():
        continue
    if not (candidate / "best.pt").is_file():
        continue
    if not (candidate / "config_resolved.yaml").is_file():
        continue
    candidates.append(candidate)

if ONLY_RUNS:
    wanted = set(ONLY_RUNS)
    missing = wanted - {path.name for path in candidates}
    if missing:
        raise RuntimeError(f"ONLY_RUNS not found under {runs_root}: {sorted(missing)}")
    found = [path for path in candidates if path.name in wanted]
else:
    raise RuntimeError(
        "ONLY_RUNS is empty. Set it to the explicit list of run folder names "
        "to evaluate (e.g. ['exp005_hybrid_fusion_seed42', ...]). Refusing to "
        "evaluate every folder under RUNS_DIR."
    )

if not found:
    raise RuntimeError(f"No run folders to evaluate under {runs_root}.")

run_image_size = {}
for run_dir in found:
    cfg = yaml.safe_load((run_dir / "config_resolved.yaml").read_text(encoding="utf-8"))
    run_image_size[run_dir.name] = int(cfg["data"]["image_size"])
required_sizes = sorted(set(run_image_size.values()))
print("runs:", [path.name for path in found])
print("image sizes:", run_image_size)


def meta_valid(processed_dir, size, spec):
        """True if data.npz + meta.json exist and match size + expected_count."""
        npz_path = processed_dir / "data.npz"
        meta_path = processed_dir / "meta.json"
        if not npz_path.is_file() or not meta_path.is_file():
            return False
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            return False
        if int(meta.get("image_size", -1)) != size:
            return False
        if int(meta.get("image_count", -1)) != int(spec.expected_count):
            return False
        return True


staged = {}  # name -> {size: data_root}
for name in DATASETS:
    spec = get_spec(name)
    staged[name] = {}
    for size in required_sizes:
        data_root = Path(LOCAL_DATA_ROOT) / f"size_{size}"
        processed_dir = data_root / "processed" / name
        npz_path = processed_dir / "data.npz"

        if name == "kvasir_seg":
            # Kvasir must be array-identical to the training cache: build it
            # from DRIVE_ZIP with build_cache.py at this size.
            zip_root = Path("/content/kvasir-zip")
            if not zip_root.is_dir():
                if not Path(DRIVE_ZIP).is_file():
                    raise FileNotFoundError(f"Drive zip not found: {DRIVE_ZIP}")
                zip_root.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(DRIVE_ZIP) as archive:
                    archive.extractall(zip_root)
            candidates = [
                p
                for p in zip_root.rglob("*")
                if p.is_dir() and (p / "images").is_dir() and (p / "masks").is_dir()
            ]
            if not candidates:
                raise FileNotFoundError(
                    f"No folder with images/ and masks/ inside {DRIVE_ZIP}"
                )
            kvasir_root = max(
                candidates,
                key=lambda p: sum(1 for _ in (p / "images").iterdir()),
            )
            cache_out = processed_dir / "data.npz"
            cache_out.parent.mkdir(parents=True, exist_ok=True)
            cmd = [
                sys.executable,
                str(REPO_DIR / "scripts" / "build_cache.py"),
                "--root",
                str(kvasir_root),
                "--size",
                str(size),
                "--out",
                str(cache_out),
            ]
            print("cache command:", " ".join(cmd), flush=True)
            result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
            print(result.stdout, end="")
            if result.returncode != 0:
                print(result.stderr, end="", flush=True)
                raise RuntimeError(f"build_cache.py failed for kvasir_seg at {size}px")
            staged[name][size] = data_root

        else:
            # External dataset: copy from Drive processed/ if valid,
            # otherwise prepare from Drive raw/.
            drive_processed = Path(DRIVE_DATA_ROOT) / "processed" / name
            if meta_valid(drive_processed, size, spec):
                processed_dir.parent.mkdir(parents=True, exist_ok=True)
                if processed_dir.exists():
                    shutil.rmtree(processed_dir)
                shutil.copytree(drive_processed, processed_dir)
                print(f"{name}: copied from Drive processed/ at {size}px", flush=True)
            elif (
                Path(DRIVE_DATA_ROOT) / "raw"
            ).is_dir() and spec.raw_dir(Path(DRIVE_DATA_ROOT) / "raw").is_dir():
                cmd = [
                    sys.executable,
                    str(REPO_DIR / "scripts" / "prepare_datasets.py"),
                    "--dataset",
                    name,
                    "--size",
                    str(size),
                    "--data-root",
                    str(Path(DRIVE_DATA_ROOT) / "raw"),
                    "--out-root",
                    str(data_root / "processed"),
                ]
                print("prep command:", " ".join(cmd), flush=True)
                result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
                print(result.stdout, end="")
                if result.returncode != 0:
                    print(result.stderr, end="", flush=True)
                    raise RuntimeError(f"prepare_datasets.py failed for {name} at {size}px")
                staged[name][size] = data_root
            else:
                raise RuntimeError(
                    f"{name} is not on Drive processed/ or raw/; unable to stage it."
                )

print("staged datasets:", {name: sorted(sizes) for name, sizes in staged.items()}, flush=True)


## Evaluate: score each run in ONLY_RUNS over DATASETS

For each run: check `best.pt` and `config_resolved.yaml` exist, write a
session config that repoints the data paths at the local staged data, then
run `python -m src.evaluate`. Every command, its stdout, its stderr and its
exit code are printed so nothing is hidden.

Before the first write, `results.csv` is copied to a timestamped backup next
to it. If `SANITY_CHECK`, each run is first evaluated on the Kvasir val
split and compared with the run's recorded best validation Dice; PASS or a
warning is printed when they differ by more than 0.003.


In [ ]:
import csv
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import yaml

sys.path.insert(0, str(REPO_DIR))

RESULTS_CSV_PATH = Path(RESULTS_CSV)
RESULTS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
results_written = False


def backup_results():
        """Copy results.csv to a timestamped backup before the first write."""
        global results_written
        if results_written or not RESULTS_CSV_PATH.is_file():
            return
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        backup = RESULTS_CSV_PATH.with_name(
            f"results.csv.{stamp}.bak"
        )
        shutil.copy2(RESULTS_CSV_PATH, backup)
        print(f"backed up results.csv -> {backup}", flush=True)
        results_written = True


def evaluate_run(run_dir, datasets, split, tta_flip, force, session_cfg):
        cmd = [
            sys.executable,
            "-m",
            "src.evaluate",
            "--run_dir",
            str(run_dir),
            "--config",
            str(session_cfg),
            "--split",
            split,
            "--checkpoint",
            "best",
            "--results_csv",
            str(RESULTS_CSV_PATH),
        ]
        for name in datasets:
            cmd += ["--dataset", name]
        if tta_flip:
            cmd.append("--tta-flip")
        if force:
            cmd.append("--force")
        print(f"\n=== {run_dir.name} [{split}] ===", flush=True)
        print("evaluate command:", " ".join(cmd), flush=True)
        result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
        print(f"evaluate exit code: {result.returncode}", flush=True)
        if result.stdout.strip():
            print(result.stdout, flush=True)
        if result.stderr.strip():
            print(result.stderr, flush=True)
        if result.returncode != 0:
            raise RuntimeError(
                f"src.evaluate failed for {run_dir.name} "
                f"(exit {result.returncode})."
            )
        return result


    # --- Session configs: repoint data paths to local staged data -------
session_dir = Path("/content/session_configs")
session_dir.mkdir(parents=True, exist_ok=True)
session_configs = {}
for run_dir in found:
    size = run_image_size[run_dir.name]
    data_root = staged["kvasir_seg"][size]
    cache_path = str(data_root / "processed" / "kvasir_seg" / "data.npz")
    ext_data_root = str(data_root)
    cfg = yaml.safe_load((run_dir / "config_resolved.yaml").read_text(encoding="utf-8"))
    cfg.setdefault("data", {})
    cfg["data"]["data_root"] = ext_data_root
    cfg["data"]["split_dir"] = str(REPO_DIR / "data" / "splits")
    cfg["data"]["cache_path"] = cache_path
    path = session_dir / f"{run_dir.name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    session_configs[run_dir] = path

print("session configs:", session_dir, flush=True)

# --- Sanity check: each run on Kvasir val --------------------------------
if SANITY_CHECK:
    print("\n--- sanity check (Kvasir val) ---", flush=True)
    for run_dir in found:
        info_path = run_dir / "run_info.json"
        recorded = None
        if info_path.is_file():
            try:
                info = json.loads(info_path.read_text(encoding="utf-8"))
                recorded = info.get("best_val_dice")
            except (json.JSONDecodeError, OSError):
                pass
        if recorded is None:
            print(f"  {run_dir.name}: no run_info.json best_val_dice; skipping", flush=True)
            continue
        backup_results()
        evaluate_run(
            run_dir,
            ["kvasir_seg"],
            "val",
            TTA_FLIP,
            FORCE,
            session_configs[run_dir],
        )
        eval_json = run_dir / f"eval_val_kvasir_seg.json"
        if eval_json.is_file():
            payload = json.loads(eval_json.read_text(encoding="utf-8"))
            val_dice = float(payload["dice_mean"])
            diff = abs(val_dice - float(recorded))
            status = "PASS" if diff <= 0.003 else "WARNING"
            print(
                f"  {run_dir.name}: val Dice {val_dice:.4f} vs recorded "
                f"{float(recorded):.4f}, diff {diff:.4f} -> {status}",
                flush=True,
            )
        else:
            print(f"  {run_dir.name}: eval_val_kvasir_seg.json not written", flush=True)

# --- Evaluate each run over DATASETS ------------------------------------
print("\n--- evaluate over DATASETS ---", flush=True)
for run_dir in found:
    backup_results()
    evaluate_run(
        run_dir,
        DATASETS,
        EVAL_SPLIT,
        TTA_FLIP,
        FORCE,
        session_configs[run_dir],
    )
print("evaluation complete", flush=True)


## Aggregate and show

Run `scripts/aggregate_results.py` with `PYTHONPATH` set to the repo root,
then print (a) the aggregate markdown and (b) per-run tables built directly
from `results.csv` (rows = run, columns = dataset, cells = test Dice / IoU).


In [ ]:
import csv
import os
import subprocess
import sys
from pathlib import Path

# --- Run aggregate_results.py ----------------------------------------
agg_cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "aggregate_results.py"),
    "--runs-dir",
    RUNS_DIR,
    "--out-dir",
    RUNS_DIR,
    "--split",
    EVAL_SPLIT,
]
if TTA_FLIP:
    agg_cmd += ["--tta", "flip"]
print("aggregate command:", " ".join(agg_cmd), flush=True)
agg_env = dict(os.environ, PYTHONPATH=str(REPO_DIR))
agg_result = subprocess.run(
    agg_cmd,
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
    env=agg_env,
)
print(f"aggregate exit code: {agg_result.returncode}", flush=True)
if agg_result.stdout.strip():
    print(agg_result.stdout, flush=True)
if agg_result.stderr.strip():
    print(agg_result.stderr, flush=True)
if agg_result.returncode != 0:
    raise RuntimeError(f"aggregate_results.py failed with exit code {agg_result.returncode}")

md_path = Path(RUNS_DIR) / "aggregate_results.md"
if md_path.is_file():
    print("\n--- aggregate markdown ---", flush=True)
    print(md_path.read_text(encoding="utf-8"))

# --- Per-run tables from results.csv -----------------------------------
print("\n--- per-run tables from results.csv ---", flush=True)
if not RESULTS_CSV_PATH.is_file():
    print("no results.csv yet", flush=True)
else:
    rows = []
    with RESULTS_CSV_PATH.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            rows.append(row)

    test_rows = [
        r for r in rows
        if r.get("split") == "test" and r.get("run_name") in {p.name for p in found}
    ]
    if not test_rows:
        print("no test rows in results.csv for the selected runs", flush=True)
    else:
        datasets_in_rows = sorted({r["dataset"] for r in test_rows})
        runs_in_rows = sorted({r["run_name"] for r in test_rows})

        def cell(rows, run, dataset, column):
            for r in rows:
                if r["run_name"] == run and r["dataset"] == dataset:
                    return r[column]
            return ""

        print("\nTest Dice by run x dataset:", flush=True)
        header = "run_name" + "".join(f" | {d}" for d in datasets_in_rows)
        print(header)
        print("-" * len(header))
        for run in runs_in_rows:
            line = run + "".join(
                f" | {cell(test_rows, run, d, 'test_dice')}" for d in datasets_in_rows
            )
            print(line)

        print("\nTest IoU by run x dataset:", flush=True)
        header = "run_name" + "".join(f" | {d}" for d in datasets_in_rows)
        print(header)
        print("-" * len(header))
        for run in runs_in_rows:
            line = run + "".join(
                f" | {cell(test_rows, run, d, 'test_iou')}" for d in datasets_in_rows
            )
            print(line)

        sl_rows = [r for r in test_rows if r.get("small_lesion_dice") not in ("", "n/a")]
        if sl_rows:
            print("\nSmall-lesion Dice by run x dataset:", flush=True)
            header = "run_name" + "".join(f" | {d}" for d in datasets_in_rows)
            print(header)
            print("-" * len(header))
            for run in runs_in_rows:
                line = run + "".join(
                    f" | {cell(test_rows, run, d, 'small_lesion_dice')}" for d in datasets_in_rows
                )
                print(line)
            print("\nSmall-lesion n by run x dataset:", flush=True)
            header = "run_name" + "".join(f" | {d}" for d in datasets_in_rows)
            print(header)
            print("-" * len(header))
            for run in runs_in_rows:
                line = run + "".join(
                    f" | {cell(test_rows, run, d, 'small_lesion_n')}" for d in datasets_in_rows
                )
                print(line)
